In [7]:
import os
import pickle

import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau,
    CSVLogger
)

In [8]:
SEED = 42

tf.random.set_seed(SEED)

In [2]:
IMG_SIZE = (224, 224)

BATCH_SIZE = 32

INITIAL_EPOCHS = 25

FINE_TUNE_EPOCHS = 30

PHASE2_EPOCHS = 20

In [5]:
dataset_path = "../../datasets/food-101/images"

print(dataset_path)
print(os.path.exists(dataset_path))

../../datasets/food-101/images
True


In [9]:
validation_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

Found 101000 files belonging to 101 classes.
Using 20200 files for validation.


In [11]:
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

Found 101000 files belonging to 101 classes.
Using 80800 files for training.


In [12]:
model = load_model("../saved_models/mobilenetv2_finetuned.keras")

print("Model Loaded Successfully")

Model Loaded Successfully


In [14]:
for i, layer in enumerate(model.layers):
    print(i, layer.name, type(layer))

0 input_layer_1 <class 'keras.src.layers.core.input_layer.InputLayer'>
1 data_augmentation <class 'keras.src.models.sequential.Sequential'>
2 mobilenetv2_1.00_224 <class 'keras.src.models.functional.Functional'>
3 global_average_pooling2d <class 'keras.src.layers.pooling.global_average_pooling2d.GlobalAveragePooling2D'>
4 dropout <class 'keras.src.layers.regularization.dropout.Dropout'>
5 dense <class 'keras.src.layers.core.dense.Dense'>


In [15]:
base_model = model.layers[2]

print(base_model.name)
print("Total Layers :", len(base_model.layers))

mobilenetv2_1.00_224
Total Layers : 154


In [16]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

print("Trainable Layers:")
print(sum([layer.trainable for layer in base_model.layers]))

Trainable Layers:
30


In [17]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model Compiled Successfully")

Model Compiled Successfully


In [18]:
checkpoint_phase2 = ModelCheckpoint(
    "../saved_models/mobilenetv2_phase2.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

In [19]:
early_stop_phase2 = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_phase2 = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

In [21]:
csv_logger_phase2 = CSVLogger(
    "../saved_models/07_training_log_phase2.csv",
    append=False
)

In [22]:
history_phase2 = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=PHASE2_EPOCHS,
    callbacks=[
        checkpoint_phase2,
        early_stop_phase2,
        reduce_lr_phase2,
        csv_logger_phase2
    ]
)

Epoch 1/20
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 0s 767ms/step - accuracy: 0.7763 - loss: 0.7941
Epoch 1: val_accuracy improved from None to 0.73515, saving model to ../saved_models/mobilenetv2_phase2.keras

Epoch 1: finished saving model to ../saved_models/mobilenetv2_phase2.keras
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 2303s 909ms/step - accuracy: 0.7765 - loss: 0.7909 - val_accuracy: 0.7351 - val_loss: 1.0173 - learning_rate: 1.0000e-05
Epoch 2/20
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 0s 800ms/step - accuracy: 0.7778 - loss: 0.7911
Epoch 2: val_accuracy improved from 0.73515 to 0.73663, saving model to ../saved_models/mobilenetv2_phase2.keras

Epoch 2: finished saving model to ../saved_models/mobilenetv2_phase2.keras
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 2372s 939ms/step - accuracy: 0.7769 - loss: 0.7901 - val_accuracy: 0.7366 - val_loss: 1.0218 - learning_rate: 1.0000e-05
Epoch 3/20
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 0s 796ms/step - accuracy: 0.7802 - loss: 0.7819
Epoch 3: val_accuracy improved from 0.73663 to 0.74